Attempting Different Feature Selection Methods

In [1]:
FIGURES_DIR = "reports"
PROCESSED_DIR = "data/processed"
RANDOM_SEED = 42

In [4]:
import pandas as pd

data = r"C:\dev\Paddy_Prediction\data\transformed\paddy_preprocessed_tree.csv"
df = pd.read_csv(data)
df.head()

,LP_Mainfield(in Tonnes),Urea_40Days,Potassh_50Days,30DRain( in mm),30DAI(in mm),30_50DRain( in mm),30_50DAI(in mm),51_70DRain(in mm),51_70AI(in mm),71_105DRain(in mm),...,Wind Direction_D61_D90_NNE,Wind Direction_D61_D90_NNW,Wind Direction_D61_D90_SE,Wind Direction_D61_D90_SW,Wind Direction_D91_D120_NW,Wind Direction_D91_D120_S,Wind Direction_D91_D120_SSE,Wind Direction_D91_D120_W,Wind Direction_D91_D120_WSW,Paddy yield(in Kg)
0,1.587832,1.587832,1.587832,1.361570,-1.361570,1.381531,-1.381531,1.235600,-1.235600,1.381531,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,35028
1,1.587832,1.587832,1.587832,1.361570,-1.361570,1.381531,-1.381531,1.235600,-1.235600,1.381531,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,35412
2,1.587832,1.587832,1.587832,-0.347370,0.347370,-0.953675,0.953675,-1.270003,1.270003,-0.953675,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,36300
3,1.587832,1.587832,1.587832,-0.347370,0.347370,-0.953675,0.953675,-1.270003,1.270003,-0.953675,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,35016
4,1.587832,1.587832,1.587832,-0.968802,0.968802,-0.486634,0.486634,-0.090896,0.090896,-0.486634,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,34044


In [5]:
#Split features target
X = df.drop(columns=["Paddy yield(in Kg)"])
y = df["Paddy yield(in Kg)"]
X, y

(      LP_Mainfield(in Tonnes)  Urea_40Days  Potassh_50Days  30DRain( in mm)  \
 0                    1.587832     1.587832        1.587832         1.361570   
 1                    1.587832     1.587832        1.587832         1.361570   
 2                    1.587832     1.587832        1.587832        -0.347370   
 3                    1.587832     1.587832        1.587832        -0.347370   
 4                    1.587832     1.587832        1.587832        -0.968802   
 ...                       ...          ...             ...              ...   
 2784                -1.890383    -1.890383       -1.890383        -0.347370   
 2785                -1.890383    -1.890383       -1.890383        -0.968802   
 2786                -1.890383    -1.890383       -1.890383        -0.968802   
 2787                -1.890383    -1.890383       -1.890383         1.361570   
 2788                -1.890383    -1.890383       -1.890383         1.361570   
 
       30DAI(in mm)  30_50DRain( in mm

### Using Pearson Correlation to check and drop redundant fatures if any

In [10]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import os

In [12]:
def remove_correlated_features(X: pd.DataFrame, threshold: float = 0.90):
    """
    Remove highly correlated numeric features to reduce redundancy.
    When two features are correlated above the threshold, the one with
    lower mean absolute correlation with all others is dropped.

    Args:
        X: Feature DataFrame.
        threshold: Absolute correlation above which a feature is dropped.

    Returns:
        Tuple (X_filtered, dropped_features).
    """
    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    corr_matrix = X[numeric_cols].corr().abs()

    # Plot full correlation heatmap
    fig, ax = plt.subplots(figsize=(20, 16))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(
        corr_matrix, mask=mask, cmap="coolwarm", center=0.5,
        linewidths=0.3, ax=ax, cbar_kws={"shrink": 0.7},
        annot=False  # too many features for annotations
    )
    ax.set_title("Pearson Correlation Heatmap (Numeric Features)", fontsize=14)
    plt.tight_layout()

    # Ensure output directory exists before saving
    os.makedirs(FIGURES_DIR, exist_ok=True)
    plt.savefig(os.path.join(FIGURES_DIR, "correlation_heatmap.png"), dpi=150)
    plt.close()
    print(f"\n[Pearson Correlation Heatmap saved]")

    # Identify and drop highly correlated pairs
    upper_tri = corr_matrix.where(mask == False)  # noqa: E712
    to_drop = set()
    for col in upper_tri.columns:
        correlated_with = upper_tri.index[upper_tri[col] > threshold].tolist()
        if correlated_with:
            # Drop the feature with higher mean correlation overall
            candidates = correlated_with + [col]
            mean_corrs = corr_matrix[candidates].mean()
            worst = mean_corrs.idxmax()
            to_drop.add(worst)

    to_drop = list(to_drop)
    print(f"[Pearson Threshold = {threshold}]")
    print(f"  Dropped {len(to_drop)} correlated features: {to_drop}")

    X_filtered = X.drop(columns=to_drop)
    return X_filtered, to_drop

In [13]:
X, _ = remove_correlated_features(X, threshold=0.90)


[Pearson Correlation Heatmap saved]
[Pearson Threshold = 0.9]
  Dropped 25 correlated features: ['30_50DAI(in mm)', 'Wind Direction_D91_D120_NW', 'Wind Direction_D61_D90_NNW', 'Wind Direction_D1_D30_ENE', 'Wind Direction_D31_D60_S', 'Inst Wind Speed_D91_D120(in Knots)', 'Wind Direction_D1_D30_SW', 'Wind Direction_D1_D30_SSE', 'Wind Direction_D91_D120_S', 'Min temp_D1_D30', 'Wind Direction_D91_D120_SSE', 'Inst Wind Speed_D31_D60(in Knots)', 'Wind Direction_D1_D30_NW', 'Wind Direction_D31_D60_WNW', 'Wind Direction_D1_D30_W', 'Wind Direction_D61_D90_NNE', 'Wind Direction_D91_D120_WSW', '71_105DRain(in mm)', 'Wind Direction_D91_D120_W', 'Wind Direction_D61_D90_SW', 'Max temp_D31_D60', '71_105DAI(in mm)', 'Trash(in bundles)', 'Wind Direction_D31_D60_NE', '30_50DRain( in mm)']
